In [ ]:
from pathlib import Path
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import PercentFormatter
import yfinance as yf

# Data

In [ ]:
tickers = ["AMZN", "AAPL", "NVDA", "GOOG"]
DATA_START = "2024-01-01"
DATA_END = "2025-01-01"

raw_prices = yf.download(
    tickers=tickers,
    start=DATA_START,
    end=DATA_END,
    interval="1d",
    auto_adjust=True,
    ignore_tz=True,
    progress=False,
)

prices = raw_prices["Close"][tickers].dropna()
prices.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
axes = axes.ravel()

company_names = {
    "AMZN": "Amazon",
    "AAPL": "Apple",
    "NVDA": "NVIDIA",
    "GOOG": "Google",
}

for ax, ticker in zip(axes, tickers):
    ax.plot(prices.index, prices[ticker], linewidth=1.5)
    ax.set_title(company_names[ticker])
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b"))

fig.suptitle("Closing Prices", fontsize=15, fontweight="bold")
fig.tight_layout()
plt.show()

In [ ]:
returns = prices.pct_change().dropna()
returns.describe().T

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(returns["AAPL"], bins=40, density=True, alpha=0.75)
ax.axvline(returns["AAPL"].mean(), linestyle="--", linewidth=1.5, label="Sample mean")
ax.set_title("Distribution of Apple Daily Returns")
ax.set_xlabel("Daily return")
ax.set_ylabel("Density")
ax.xaxis.set_major_formatter(PercentFormatter(1.0))
ax.grid(axis="y", linestyle="--", alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False)

plt.tight_layout()
plt.show()

In [ ]:
asset_data = np.asarray(returns, order="C")
num_samples, num_assets = asset_data.shape
header = list(returns.columns)

matrix_budget = np.ones(num_assets)
matrix_means = np.mean(asset_data, axis=0)
matrix_cov = np.cov(asset_data, rowvar=False)
matrix_assets = asset_data

print(f"Number of scenarios: {num_samples}")
print(f"Number of assets:    {num_assets}")
print("Asset order:        ", header)
print("Mean returns:       ", matrix_means)

# Mean-variance portfolio optimization

## Long-only portfolio

\begin{aligned}
\min_w\quad & w^\top\Sigma w\\
\text{s.t.}\quad & \mu^\top w\geq \mu^*,\\
& \mathbf 1^\top w=1,\\
& w\geq 0.
\end{aligned}


Here $w^\top\Sigma w$ is portfolio variance, $\mu^\top w$ is expected daily return, and $\mu^*$ is the required expected return.

In [ ]:
target_return = 0.002  # 0.20% expected daily return

In [ ]:
import gurobipy as gp
from gurobipy import GRB

model = gp.Model("Portfolio Optimization")
model.Params.OutputFlag = 0

w = model.addMVar(
    shape=num_assets,
    lb=0, # Lower bound of 0 for weights (no short selling)
    ub=GRB.INFINITY,
    vtype=GRB.CONTINUOUS,
    name=header,
)

model.setObjective(w @ matrix_cov @ w, GRB.MINIMIZE) # Minimize portfolio variance
model.addConstr(matrix_budget @ w == 1) # Budget constraint
model.addConstr(matrix_means @ w >= target_return) # Expected return constraint

model.optimize()

optimal_weights = pd.Series(w.X, index=header, name="Optimal Weights")
print(optimal_weights)